In [ ]:
import zipfile
import json
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz

In [15]:
def zip_to_sparse_matrix(zip_path: str):
    """
    Lee un ZIP con JSONs de playlists de Spotify y devuelve una sparse matrix (CSR).
    
    Filas = playlists (por pid), Columnas = canciones (por track_uri).
    Valor 1 si la canción aparece en la playlist, 0 en caso contrario.
    Las playlists vacías se conservan como filas de ceros.
    
    UNA SOLA PASADA por el ZIP (optimizado para archivos grandes).
    
    Returns:
        matrix: csr_matrix de forma (n_playlists, n_tracks)
        pid_to_row: dict {pid: row_index}
        uri_to_col: dict {track_uri: col_index}
    """
    rows, cols = [], []
    pid_to_row = {}
    uri_to_col = {}
    next_pid = 0
    next_uri = 0

    with zipfile.ZipFile(zip_path, "r") as zipf:
        filenames = [f for f in zipf.namelist() if f.endswith(".json")]
        total = len(filenames)
        for i, file in enumerate(filenames):
            with zipf.open(file) as f:
                data = json.loads(f.read())
                for playlist in data["playlists"]:
                    pid = playlist["pid"]
                    if pid not in pid_to_row:
                        pid_to_row[pid] = next_pid
                        next_pid += 1
                    row = pid_to_row[pid]

                    for track in playlist.get("tracks", []):
                        uri = track["track_uri"]
                        if uri not in uri_to_col:
                            uri_to_col[uri] = next_uri
                            next_uri += 1
                        rows.append(row)
                        cols.append(uri_to_col[uri])

            if (i + 1) % 100 == 0 or (i + 1) == total:
                print(f"\r  Procesados {i + 1}/{total} archivos JSON", end="", flush=True)

    print()  # salto de línea tras el progreso

    n_playlists = len(pid_to_row)
    n_tracks = len(uri_to_col)
    values = np.ones(len(rows), dtype=np.int8)

    matrix = csr_matrix((values, (rows, cols)), shape=(n_playlists, n_tracks))

    print(f"Shape: {matrix.shape} ({n_playlists} playlists x {n_tracks} tracks)")
    print(f"Non-zero elements: {matrix.nnz}")
    print(f"Sparsity: {(1 - matrix.nnz / (n_playlists * n_tracks)) * 100:.4f}%")

    return matrix, pid_to_row, uri_to_col

# Test

In [ ]:
matrix_test, pid_map_test, uri_map_test = zip_to_sparse_matrix("datos/spotify_test_playlists.zip")
save_npz("sparse_matrix_test.npz", matrix_test, compressed=False)
print("Guardada en 'sparse_matrix_test.npz'")

# Train

In [14]:
matrix_train, pid_map_train, uri_map_train = zip_to_sparse_matrix("datos/spotify_train_dataset.zip")
save_npz("sparseMatrixTrain.npz", matrix_train, compressed=False)
print("Guardada en 'sparse_matrix_train.npz'")

KeyboardInterrupt: 

hay que guardar también las playlist vacías?

hay que guardar también la información de las playlists

In [ ]:
zip_path = "datos/spotify_test_playlists.zip"

playlist_id, track_id, value = [], [], []

# Open the ZIP file
with zipfile.ZipFile(zip_path, "r") as zipf:
    # Iterate over each file in the ZIP
    for file in zipf.namelist():
        # Check if the file is a JSON
        if file.endswith(".json"):
            # Read the content of the JSON file
            with zipf.open(file) as f:
                file_data = json.loads(f.read())
                # Extract row, column, and value data
                for playlist in file_data["playlists"]:
                    if playlist["tracks"]:
                        for tracks in playlist["tracks"]:
                            playlist_id.append(playlist["pid"])
                            track_id.append(tracks["track_uri"])
                            value.append(1)

In [5]:
def string_to_int(lista: list):
    unique_values = sorted(set(lista))
    mapeo = {valor: indice for indice, valor in enumerate(unique_values)}
    lista_int = [mapeo[s] for s in lista]

    return lista_int, mapeo

In [40]:
playlist_id_int, playlist_id_mapeo = string_to_int(playlist_id)
track_id_int, track_id_mapeo = string_to_int(track_id)

In [41]:
matrix = csr_matrix((value, (playlist_id_int, track_id_int)), shape=(len(playlist_id_int), len(track_id_int)))

In [ ]:
# Save the matrix to a .npz file
save_npz("sparse_matrix_test.npz", matrix, compressed=False)
print("CSR matrix created and saved to 'sparse_matrix_test.npz'.")

In [45]:
# Load the matrix (keep it sparse!)
matrix2 = load_npz("sparse_matrix_test.npz")
print("\nMatrix loaded from 'sparse_matrix_test.npz':")
print(f"Shape: {matrix2.shape}")
print(f"Non-zero elements: {matrix2.nnz}")
print(f"Sparsity: {(1 - matrix2.nnz / (matrix2.shape[0] * matrix2.shape[1])) * 100:.4f}%")
print(f"Data type: {matrix2.dtype}")

# View a small sample (e.g., first 10x10 block)
print("\nFirst 10x10 block:")
print(matrix2[:10, :1000].toarray())


Matrix loaded from 'sparse_matrix_test.npz':
Shape: (697567, 697567)
Non-zero elements: 697567
Sparsity: 99.9999%
Data type: int64

First 10x10 block:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


# ahora lo mismo pero para los de train

In [4]:
zip_path = "datos/spotify_train_dataset.zip"

playlist_id, track_id, value = [], [], []

# Open the ZIP file
with zipfile.ZipFile(zip_path, "r") as zipf:
    # Iterate over each file in the ZIP
    for file in zipf.namelist():
        # Check if the file is a JSON
        if file.endswith(".json"):
            # Read the content of the JSON file
            with zipf.open(file) as f:
                file_data = json.loads(f.read())
                # Extract row, column, and value data
                for playlist in file_data["playlists"]:
                    if playlist["tracks"]:
                        for tracks in playlist["tracks"]:
                            playlist_id.append(playlist["pid"])
                            track_id.append(tracks["track_uri"])
                            value.append(1)
                    # else:
                    #     playlist_id.append(playlist["pid"])
                    #     track_id.append(tracks["track_uri"])
                    #     value.append(0)

In [6]:
playlist_id_int, playlist_id_mapeo = string_to_int(playlist_id)
track_id_int, track_id_mapeo = string_to_int(track_id)

In [12]:
lista = []
if lista:
    print('hola')

In [7]:
n_playlists = len(playlist_id_mapeo)
n_tracks = len(track_id_mapeo)

matrix = csr_matrix(
    (value, (playlist_id_int, track_id_int)),
    shape=(n_playlists, n_tracks)
)

In [8]:
# Save the matrix to a .npz file
save_npz("sparse_matrix_train.npz", matrix, compressed=False)
print("CSR matrix created and saved to 'sparse_matrix_train.npz'.")

CSR matrix created and saved to 'sparse_matrix_train.npz'.


In [9]:
# Load the matrix (keep it sparse!)
matrix2 = load_npz("sparse_matrix_train.npz")
print("\nMatrix loaded from 'sparse_matrix_test.npz':")
print(f"Shape: {matrix2.shape}")
print(f"Non-zero elements: {matrix2.nnz}")
print(f"Sparsity: {(1 - matrix2.nnz / (matrix2.shape[0] * matrix2.shape[1])) * 100:.4f}%")
print(f"Data type: {matrix2.dtype}")

# View a small sample (e.g., first 10x10 block)
print("\nFirst 10x10 block:")
print(matrix2[:10, :1000].toarray())


Matrix loaded from 'sparse_matrix_test.npz':
Shape: (990000, 2262292)
Non-zero elements: 64767209
Sparsity: 99.9971%
Data type: int64

First 10x10 block:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
